### Stage 1 

##### Pipeline to extract information from website Glassdoor - Web Scrapping

In [ ]:
import asyncio
import os
import random
import pandas as pd
from playwright.async_api import async_playwright, Playwright

async def scrape_microsoft_reviews(playwright: Playwright):
    # Edge profile configuration to keep login information - Please change User Name to your own
    user_data_dir = os.path.join(os.environ['LOCALAPPDATA'], r'Microsoft\Edge\User Name')
    
    context = await playwright.chromium.launch_persistent_context(
        user_data_dir,
        channel="msedge",
        headless=False,
        args=["--disable-blink-features=AutomationControlled"],
        ignore_default_args=["--enable-automation"]
    )

    page = context.pages[0] if context.pages else await context.new_page()
    url = "https://www.glassdoor.com.mx/Opiniones/Microsoft-Opiniones-E1651.htm"
    
    reviews_list = []

    try:
        print(f"Navigating to: {url}...")
        await page.goto(url, wait_until="domcontentloaded", timeout=60000)

        # Pause to simulate human behaviour
        await asyncio.sleep(random.uniform(4, 7))

        print("If captcha appears, please resolve it manually in the website window...")
        
        # wait until the principal container is visible
        try:
            await page.wait_for_selector('[data-test="review-detail"]', timeout=30000)
        except Exception:
            print("Error: No reviews detected. Please review if captcha blocked the access.")
            return

        # The following code block will locate all reviews containers
        reviews = await page.locator('[data-test="review-detail"]').all()
        print(f"Found {len(reviews)} reviews to process.")

        # This loop iterates through all the posted reviews
        for i, review in enumerate(reviews):
            try:
                # Pause between each review to not timeout
                await asyncio.sleep(random.uniform(2, 4))

                # The following are all the elements to export for the data file
                title_loc = review.locator('.heading_Heading__aomVx.heading_Level3__py3_P')
                rating_loc = review.locator('.ReviewRating_ratingLabel__uZhnh')
                jposition_loc = review.locator('.ContentAvatarTags_avatarLabel__Nb7Nh')
                e_status_loc = review.locator('.text-with-icon_TextWithIcon__Fs1BS')
                pros_loc = review.locator('[data-test="review-text-PROS"]')
                cons_loc = review.locator('[data-test="review-text-CONS"]')

                # Secure extraction of text items
                title = await title_loc.inner_text(timeout=2000) if await title_loc.count() > 0 else "N/A"
                rating = await rating_loc.inner_text(timeout=2000) if await rating_loc.count() > 0 else "N/A"
                jposition = await jposition_loc.inner_text(timeout=2000) if await jposition_loc.count() > 0 else "N/A"
                e_status = await e_status_loc.first.inner_text(timeout=2000) if await e_status_loc.count() > 0 else "N/A"
                advantages = await pros_loc.inner_text(timeout=2000) if await pros_loc.count() > 0 else "N/A"
                desadvantages = await cons_loc.inner_text(timeout=2000) if await cons_loc.count() > 0 else "N/A"

                reviews_list.append({
                    "Title": title.strip(),
                    "Rating": rating.strip(),
                    "Job Position": jposition.strip(),
                    "Employee Status": e_status.strip(),
                    "Advantages": advantages.strip(),
                    "Desadvantages": desadvantages.strip()
                })
                print(f" [{i+1}/{len(reviews)}] Extracted: {title[:30]}...")

            except Exception as e:
                print(f" Error omitted in review {i+1}: {e}")
                continue

        # CSV file creation
        if reviews_list:
            df = pd.DataFrame(reviews_list)
            df.to_csv("reviews_microsoft.csv", index=False, encoding='utf-8-sig')
            print(f"\n File 'microsoft_resenas.csv' created succesfully with {len(df)} rows.")
        else:
            print("\n No reviews were extracted. Please verify the classes selectors.")

    finally:
        await asyncio.sleep(2)
        await context.close()

async def main():
    async with async_playwright() as playwright:
        await scrape_microsoft_reviews(playwright)

if __name__ == "__main__":
    asyncio.run(main())